In [54]:
import torch
import torch.nn as nn

In [55]:
batch_size = 64
sequence_len = 64
embedding_dim = 32
layer_size = 256
steps = 5000

In [56]:
# input & output initialization
with open("inputs-shakespeare.txt", "r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(chars)}
itoc = dict(enumerate(chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, sequence_len, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(0, sequence_len, 1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, sequence_len, sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(0, sequence_len, sequence_len)

In [57]:
# model declaration
class RecurrentNeuralNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_dim, layer_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.input_linear = nn.Linear(in_features=embedding_dim, out_features=layer_size)
        self.hidden_linear = nn.Linear(in_features=layer_size, out_features=layer_size)
        self.output_linear = nn.Linear(in_features=layer_size, out_features=vocab_size)        

    def forward(self, input, h=None):
        batch_size = input.size(0)
        if h is None:
            h = torch.zeros((batch_size, layer_size))
        logits = []
        for i in range(input.size(1)):
            ch = input[:, i]
            new_h = torch.tanh(self.input_linear(self.embedding(ch)) + self.hidden_linear(h))
            logits.append(self.output_linear(new_h))
            h = new_h
        return torch.stack(logits, dim=1), h

In [58]:
# model initialization
model = RecurrentNeuralNetwork(vocab_size, embedding_dim, layer_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [59]:
# training
for step in range(steps):
    optimizer.zero_grad()
    
    ix = torch.randint(0, len(x_train), (batch_size,))
    logits_train, _ = model(x_train[ix])
    loss_train = nn.functional.cross_entropy(logits_train.view(-1, vocab_size), y_train[ix].view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            logits_val, _ = model(x_val)
            loss_val = nn.functional.cross_entropy(logits_val.view(-1, vocab_size), y_val.view(-1))
            print(f"Step {step}: train loss = {loss_train.item()}, val loss = {loss_val.item()}")

    loss_train.backward()
    optimizer.step()

Step 0: train loss = 4.236037254333496, val loss = 4.2415313720703125
Step 1000: train loss = 1.5039849281311035, val loss = 1.7345678806304932
Step 2000: train loss = 1.5004528760910034, val loss = 1.666714072227478
Step 3000: train loss = 1.4609907865524292, val loss = 1.643480658531189
Step 4000: train loss = 1.3997325897216797, val loss = 1.6488707065582275
Step 4999: train loss = 1.4159098863601685, val loss = 1.6346908807754517


In [60]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000
    h = None

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)            

    while len(output) < max_output_length:
        logits, h = model(i, h)

        P = torch.softmax(logits[0, -1, :], dim=0)
        j = torch.multinomial(P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[output[-1]]]], dtype=torch.long)

In [61]:
print(''.join(output))

ROMEO:

LURET:
A house,
Unfinate by mistre withally wonder is last.
O, let--I
I are with this passage hand behold infectors to Romeo's fements, thy hardly had
And leave you your faith ne'er thou will we.

HERMIOHMASY VIRGUS:
A haven by thweckny service with sleeper un, and and quits, the soul?

LUCIO:
The seat burning shadon, just mine ear-deed; what would, making, she hath dropprain,
Rewell,
The
earle and than it is none.

ROME:
Underdre, and speaks, let you so.

DUKE OF MALELLA:
Sir. are you, by that entermary eye of heaven out off not the winthem herefore come, sad with swiness in;
To-min: thus is mo; I dost have thoughts
of nothing in't your neck my with thee form'd; for mine what give him old-place Claudio not usush thee thou undergmoniste; the land with you.

ANGELID:
Again, which hopreners of you?

ROMEO:
Whysence you are it you, kind office of the execcision of this times for his dukedom when I depent you wounding penereel I did I will strike hour that kill the crumined, would 